# Skip-safety metrics: natural (s2) + medical (s4) summary table

Per combination: Spearman $\rho$ between each metric (sign-corrected so higher = safer) and the true score of every skip,
and top-5 damage = baseline minus the mean true score of the metric's 5 highest-ranked skips (pp).
Each table cell is the mean over combinations. Natural = accuracy, medical = AUROC.

Every combination's candidates are skips of 1-4 blocks, and longer skips cost far more, so the all-span $\rho$ is
mostly "does the metric grow with span length". Two additions separate that out:
- **Within-length $\rho$** ($\rho_\text{len}$): Spearman computed separately among the skips of each length (1, 2, 3, 4 blocks),
  then averaged over the four lengths -- how well a metric picks *which* blocks once the budget is fixed.
- **Fewest blocks** baseline: rank by span length alone. Its top 5 are a random 5 of the single-block skips, so its
  expected damage is the mean damage of a random single-block skip.

Reads the stage-1 CSVs (`output_level_eval_*.csv`) and the accuracy CSVs directly, so `build_group_stats.py` is not needed.

In [1]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

ROOT = Path.cwd().resolve()
while not (ROOT / "experiments").is_dir():
    ROOT = ROOT.parent
S2, S4 = ROOT / "experiments" / "4_1-4_2_choosing_blocks" / "natural", ROOT / "experiments" / "4_1-4_2_choosing_blocks" / "medical"

NATURAL = [(f"{m}_{d}", S2 / "analysis" / f"output_level_eval_s2_{m}_{d}.csv",
            S2 / "accuracies" / f"accuracies_results_{m}_{d}.csv", "accuracy")
           for m in ["deitsmall", "vitsmall", "dinov2base", "vitlarge"] for d in ["cifar100", "imagenet1k"]]

# Chest: the 14-label ChestX-ray14 task (not the single-label one) + pneumonia. Histo: PCam + TCGA-UT.
MEDICAL = [(f"{m}_{d}", S4 / "analysis" / f"output_level_eval_s4_{m}_{d}.csv",
            S4 / "accuracies" / "combined" / f"accuracies_results_{m}_{d}.csv", "auroc")
           for models, datasets in [(["dinov2base", "raddino", "raddinomaira2"], ["chestxray14multilabel", "chestxraypneumonia"]),
                                    (["vitlarge", "phikonv2"], ["patchcamelyon", "tcgaut"])]
           for m in models for d in datasets]

# Combos whose stage-1 CSV is out of date (none at present).
EXCLUDE = set()

METRICS = {
    "angular_dist": "Angular distance",
    "local_cka": "Local CKA",
    "cka_embed": "Global CKA",
    "cosine_sim": "Cosine similarity",
    "toast_linear_error_alltok": "Linear approx.\\ error (TOAST)",
    "toast_linear_error_alltok_heldout": "Linear approx.\\ error (TOAST, test set)",
    "mse": "MSE",
}
HEAD_METRICS = {"orig_head_accuracy": "Pretrained head accuracy", "kl_div": "KL divergence"}
DISSIMILARITY = {"angular_dist", "toast_linear_error_alltok", "toast_linear_error_alltok_heldout", "mse", "kl_div"}  # higher raw value = less safe

In [2]:
def _span(approx_layer):
    parsed = ast.literal_eval(approx_layer)
    return tuple(parsed[0]) if len(parsed) == 1 else None


def evaluate_combo(eval_csv, acc_csv, target, metrics):
    ev, acc = pd.read_csv(eval_csv), pd.read_csv(acc_csv)
    baseline = acc.loc[acc["approx_layer"] == "[]", f"{target}_mean"].iloc[0]
    acc["span"] = acc["approx_layer"].map(_span)
    truth = dict(zip(acc["span"], acc[f"{target}_mean"]))
    ev["truth"] = [truth.get(s) for s in zip(ev["skip_from"], ev["skip_to"])]
    ev = ev.dropna(subset=["truth"])
    length = ev["skip_to"] - ev["skip_from"]

    def rho_within_length(score):
        return np.mean([spearmanr(score[length == n], ev.loc[length == n, "truth"])[0]
                        for n in sorted(length.unique()) if (length == n).sum() >= 5])

    row = {"Fewest blocks_rho": spearmanr(-length, ev["truth"])[0],
           "Fewest blocks_dmg": (baseline - ev.loc[length == length.min(), "truth"].mean()) * 100,
           "Random skip (any length)_dmg": (baseline - ev["truth"].mean()) * 100,
           "Oracle (best 5 skips)_dmg": (baseline - ev["truth"].nlargest(5).mean()) * 100}
    for m in metrics:
        if m not in ev or ev[m].isna().all():
            continue
        score = -ev[m] if m in DISSIMILARITY else ev[m]
        row[f"{m}_rho"] = spearmanr(score, ev["truth"])[0]
        row[f"{m}_rholen"] = rho_within_length(score)
        row[f"{m}_dmg"] = (baseline - ev.loc[score.nlargest(5).index, "truth"].mean()) * 100
    return row


def evaluate(combos, metrics):
    rows = {name: evaluate_combo(e, a, t, metrics) for name, e, a, t in combos if name not in EXCLUDE}
    return pd.DataFrame(rows).T


BASELINE_ROWS = ["Fewest blocks", "Random skip (any length)", "Oracle (best 5 skips)"]


def summary(per_combo, metrics):
    mean = per_combo.mean()
    keys = list(metrics) + BASELINE_ROWS
    return pd.DataFrame({k: [mean.get(f"{m}_{k}", np.nan) for m in keys] for k in ("rho", "rholen", "dmg")},
                        index=list(metrics.values()) + BASELINE_ROWS)


nat = evaluate(NATURAL, METRICS)
med = evaluate(MEDICAL, METRICS)
print(f"natural: {len(nat)} combos, medical: {len(med)} combos (excluded: {sorted(EXCLUDE) or 'none'})")
table = pd.concat({"Natural": summary(nat, METRICS), "Medical": summary(med, METRICS)}, axis=1)
table.round(2)

natural: 8 combos, medical: 10 combos (excluded: none)


Natural               Medical         \
                                            rho rholen    dmg     rho rholen   
Angular distance                           0.85   0.50   1.75    0.46   0.35   
Local CKA                                  0.66   0.23   4.05    0.47   0.31   
Global CKA                                 0.91   0.71   1.66    0.34   0.06   
Cosine similarity                          0.91   0.73   1.53    0.32   0.04   
Linear approx.\ error (TOAST)              0.75   0.35   4.57    0.45   0.38   
Linear approx.\ error (TOAST, test set)    0.75   0.35   4.57    0.45   0.38   
MSE                                        0.91   0.73   1.53    0.32   0.04   
Fewest blocks                              0.81    NaN   4.00    0.32    NaN   
Random skip (any length)                    NaN    NaN  14.39     NaN    NaN   
Oracle (best 5 skips)                       NaN    NaN   1.14     NaN    NaN   

                                               
                                          dmg  
Angular distance                        -0.10  
Local CKA                               -0.07  
Global CKA                               0.04  
Cosine similarity                        0.00  
Linear approx.\ error (TOAST)           -0.03  
Linear approx.\ error (TOAST, test set) -0.03  
MSE                                      0.00  
Fewest blocks                            0.06  
Random skip (any length)                 0.33  
Oracle (best 5 skips)                   -0.57

In [3]:
tabledf = nat.iloc[:4]
tabledf["toast_linear_error_alltok_rholen"].mean()


np.float64(0.36797889610389606)

## LaTeX

In [4]:
# Columns in the LaTeX table. Medical damage is left out: its spread across metrics (~0.2 pp) is within
# seed noise (see the noise section below). All-span rho is shown but not bolded, since length drives it.
COLS = [("Natural", "rho"), ("Natural", "rholen"), ("Natural", "dmg"), ("Medical", "rho"), ("Medical", "rholen")]
BOLD = {("Natural", "rholen"): "max", ("Natural", "dmg"): "min", ("Medical", "rholen"): "max"}
LATEX_ROWS = list(METRICS.values()) + ["Fewest blocks", "Oracle (best 5 skips)"]
LATEX_NAMES = {"Fewest blocks": "Fewest blocks (random single block)"}


def latex(table, n_nat, n_med):
    metric_rows = list(METRICS.values())
    best = {}
    for col, how in BOLD.items():
        shown = table.loc[metric_rows, col].round(2)
        best[col] = set(shown.index[shown == (shown.max() if how == "max" else shown.min())])

    def cell(name, col):
        v = table.loc[name, col]
        if pd.isna(v):
            return "--"
        s = f"{v:.2f}"
        return f"\\textbf{{{s}}}" if name in best.get(col, set()) else s

    lines = [r"\begin{table}[t]", r"\centering", r"\small", r"\begin{tabular}{lccccc}", r"\toprule",
             r" & \multicolumn{3}{c}{Natural (Sec.~\ref{metric_nat})} & \multicolumn{2}{c}{Medical} \\",
             r"\cmidrule(lr){2-4}\cmidrule(lr){5-6}",
             r"Metric & $\rho\uparrow$ & $\rho_{\text{len}}\uparrow$ & Damage $\downarrow$ "
             r"& $\rho\uparrow$ & $\rho_{\text{len}}\uparrow$ \\", r"\midrule"]
    for name in LATEX_ROWS:
        if name == "Fewest blocks":
            lines.append(r"\midrule")
        lines.append(f"{LATEX_NAMES.get(name, name):36s} & " + " & ".join(cell(name, c) for c in COLS) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}",
              rf"\caption{{Mean Spearman $\rho$ over all skips, mean within-length $\rho_{{\text{{len}}}}$ "
              rf"(computed among skips of the same length, then averaged over lengths 1--4), and top-5 damage (pp), "
              rf"over {n_nat} natural-image combinations (accuracy) and {n_med} medical ones (AUROC). "
              rf"TOAST: least-squares error over all tokens, scored on the images the map was fit on; "
              rf"TOAST, test set: the same map scored on a disjoint set of images. "
              rf"Fewest blocks ranks skips by length alone; its top five are a random five single-block skips. "
              rf"Oracle: damage of the five best skips. Medical top-5 damage is omitted because every metric's value "
              rf"lies within seed noise of every other's.}}",
              r"\label{tab:metrics_nat_med}", r"\end{table}"]
    return "\n".join(lines)


print(latex(table, len(nat), len(med)))

\begin{table}[t]
\centering
\small
\begin{tabular}{lccccc}
\toprule
 & \multicolumn{3}{c}{Natural (Sec.~\ref{metric_nat})} & \multicolumn{2}{c}{Medical} \\
\cmidrule(lr){2-4}\cmidrule(lr){5-6}
Metric & $\rho\uparrow$ & $\rho_{\text{len}}\uparrow$ & Damage $\downarrow$ & $\rho\uparrow$ & $\rho_{\text{len}}\uparrow$ \\
\midrule
Angular distance                     & 0.85 & 0.50 & 1.75 & 0.46 & 0.35 \\
Local CKA                            & 0.66 & 0.23 & 4.05 & 0.47 & 0.31 \\
Global CKA                           & 0.91 & 0.71 & 1.66 & 0.34 & 0.06 \\
Cosine similarity                    & 0.91 & \textbf{0.73} & \textbf{1.53} & 0.32 & 0.04 \\
Linear approx.\ error (TOAST)        & 0.75 & 0.35 & 4.57 & 0.45 & \textbf{0.38} \\
Linear approx.\ error (TOAST, test set) & 0.75 & 0.35 & 4.57 & 0.45 & \textbf{0.38} \\
MSE                                  & 0.91 & \textbf{0.73} & \textbf{1.53} & 0.32 & 0.04 \\
\midrule
Fewest blocks (random single block)  & 0.81 & -- & 4.00 & 0.32 & -- \\
Oracle (be

## Paired within-length comparisons

For each metric, in how many combinations its $\rho_\text{len}$ beats angular distance's and cosine similarity's,
and the mean difference. The sign-test p-value treats combinations as independent, which they are not quite
(the same models and datasets recur), so read it as a rough guide.

In [5]:
from scipy.stats import binomtest


def paired(per_combo, metrics, refs=("angular_dist", "cosine_sim")):
    rows = []
    for ref in refs:
        for m in metrics:
            if m == ref:
                continue
            d = (per_combo[f"{m}_rholen"] - per_combo[f"{ref}_rholen"]).dropna()
            wins = int((d > 0).sum())
            rows.append({"metric": METRICS[m], "vs": METRICS[ref], "mean diff": d.mean(),
                         "better in": f"{wins}/{len(d)}", "sign-test p": binomtest(wins, len(d)).pvalue})
    return pd.DataFrame(rows).set_index(["vs", "metric"])


for label, df in [("Natural", nat), ("Medical", med)]:
    print(label)
    display(paired(df, METRICS).round(2))

Natural


mean diff  \
vs                metric                                               
Angular distance  Local CKA                                    -0.27   
                  Global CKA                                    0.21   
                  Cosine similarity                             0.23   
                  Linear approx.\ error (TOAST)                -0.15   
                  Linear approx.\ error (TOAST, test set)      -0.15   
                  MSE                                           0.23   
Cosine similarity Angular distance                             -0.23   
                  Local CKA                                    -0.50   
                  Global CKA                                   -0.02   
                  Linear approx.\ error (TOAST)                -0.38   
                  Linear approx.\ error (TOAST, test set)      -0.38   
                  MSE                                          -0.00   

                                                          better in  \
vs                metric                                              
Angular distance  Local CKA                                     4/8   
                  Global CKA                                    5/8   
                  Cosine similarity                             5/8   
                  Linear approx.\ error (TOAST)                 3/8   
                  Linear approx.\ error (TOAST, test set)       3/8   
                  MSE                                           6/8   
Cosine similarity Angular distance                              3/8   
                  Local CKA                                     2/8   
                  Global CKA                                    3/8   
                  Linear approx.\ error (TOAST)                 2/8   
                  Linear approx.\ error (TOAST, test set)       2/8   
                  MSE                                           3/8   

                                                           sign-test p  
vs                metric                                                
Angular distance  Local CKA                                       1.00  
                  Global CKA                                      0.73  
                  Cosine similarity                               0.73  
                  Linear approx.\ error (TOAST)                   0.73  
                  Linear approx.\ error (TOAST, test set)         0.73  
                  MSE                                             0.29  
Cosine similarity Angular distance                                0.73  
                  Local CKA                                       0.29  
                  Global CKA                                      0.73  
                  Linear approx.\ error (TOAST)                   0.29  
                  Linear approx.\ error (TOAST, test set)         0.29  
                  MSE                                             0.73

Medical


mean diff  \
vs                metric                                               
Angular distance  Local CKA                                    -0.04   
                  Global CKA                                   -0.29   
                  Cosine similarity                            -0.31   
                  Linear approx.\ error (TOAST)                 0.02   
                  Linear approx.\ error (TOAST, test set)       0.02   
                  MSE                                          -0.31   
Cosine similarity Angular distance                              0.31   
                  Local CKA                                     0.27   
                  Global CKA                                    0.02   
                  Linear approx.\ error (TOAST)                 0.33   
                  Linear approx.\ error (TOAST, test set)       0.33   
                  MSE                                           0.00   

                                                          better in  \
vs                metric                                              
Angular distance  Local CKA                                    6/10   
                  Global CKA                                   6/10   
                  Cosine similarity                            6/10   
                  Linear approx.\ error (TOAST)                7/10   
                  Linear approx.\ error (TOAST, test set)      7/10   
                  MSE                                          5/10   
Cosine similarity Angular distance                             4/10   
                  Local CKA                                    7/10   
                  Global CKA                                   6/10   
                  Linear approx.\ error (TOAST)                5/10   
                  Linear approx.\ error (TOAST, test set)      5/10   
                  MSE                                          3/10   

                                                           sign-test p  
vs                metric                                                
Angular distance  Local CKA                                       0.75  
                  Global CKA                                      0.75  
                  Cosine similarity                               0.75  
                  Linear approx.\ error (TOAST)                   0.34  
                  Linear approx.\ error (TOAST, test set)         0.34  
                  MSE                                             1.00  
Cosine similarity Angular distance                                0.75  
                  Local CKA                                       0.34  
                  Global CKA                                      0.75  
                  Linear approx.\ error (TOAST)                   1.00  
                  Linear approx.\ error (TOAST, test set)         1.00  
                  MSE                                             0.34

## Pretrained-head metrics (natural, ImageNet-1k only)

Only DeiT-S, ViT-S and ViT-L ship an ImageNet head. `orig_head_accuracy` is computed on held-out *training* images
(never the test images behind the ground truth). Angular distance is shown on the same combos for reference.

In [6]:
head = evaluate(NATURAL, {**METRICS, **HEAD_METRICS})
head = head[head["orig_head_accuracy_rho"].notna()]
print("combos:", list(head.index))
summary(head, {"angular_dist": "Angular distance", "cosine_sim": "Cosine similarity", **HEAD_METRICS}).iloc[:-3].round(2)

combos:

 ['deitsmall_imagenet1k', 'vitsmall_imagenet1k', 'vitlarge_imagenet1k']


,rho,rholen,dmg
Angular distance,0.83,0.52,2.24
Cosine similarity,0.96,0.82,1.75
Pretrained head accuracy,0.96,0.80,1.57
KL divergence,0.96,0.76,1.91


## Seed noise (s2/s4)

Ground truth is the mean over 3 seeds. Per combination: the seed SD of one skip's score, and the approximate SD of a
top-5 mean (one-skip SD / $\sqrt{5}$; optimistic, since seeds share images across skips). Differences in top-5 damage
smaller than about twice the top-5 SD should not be read as real.

In [7]:
def noise(combos):
    rows = {}
    for name, _, acc_csv, target in combos:
        if name in EXCLUDE:
            continue
        acc = pd.read_csv(acc_csv)
        sd = acc.loc[acc["approx_layer"] != "[]", f"{target}_std"] * 100
        rows[name] = {"baseline SD (pp)": acc.loc[acc["approx_layer"] == "[]", f"{target}_std"].iloc[0] * 100,
                      "one-skip SD (pp)": sd.mean(), "top-5 mean SD (pp)": sd.mean() / np.sqrt(5)}
    df = pd.DataFrame(rows).T
    df.loc["MEAN"] = df.mean()
    return df


display(noise(NATURAL).round(2))
noise(MEDICAL).round(2)

,baseline SD (pp),one-skip SD (pp),top-5 mean SD (pp)
deitsmall_cifar100,0.23,0.28,0.12
deitsmall_imagenet1k,0.10,0.28,0.12
vitsmall_cifar100,0.33,0.36,0.16
vitsmall_imagenet1k,0.26,0.30,0.13
dinov2base_cifar100,0.20,0.56,0.25
dinov2base_imagenet1k,0.37,0.45,0.20
vitlarge_cifar100,0.41,0.53,0.24
vitlarge_imagenet1k,0.32,0.41,0.18
MEAN,0.28,0.39,0.18


,baseline SD (pp),one-skip SD (pp),top-5 mean SD (pp)
dinov2base_chestxray14multilabel,0.20,0.24,0.11
dinov2base_chestxraypneumonia,0.40,0.40,0.18
raddino_chestxray14multilabel,0.21,0.17,0.08
raddino_chestxraypneumonia,0.01,0.01,0.01
raddinomaira2_chestxray14multilabel,0.76,0.57,0.26
raddinomaira2_chestxraypneumonia,0.17,0.25,0.11
vitlarge_patchcamelyon,0.36,0.48,0.21
vitlarge_tcgaut,0.10,0.12,0.06
phikonv2_patchcamelyon,0.21,0.21,0.09
phikonv2_tcgaut,0.03,0.03,0.01


## Seed noise for the translator-transfer table (s5, Sec. 4.4)

Each seed fixes the training images and the head's initialisation, and both the own-fit and the transfer run under
that seed use them, so the gap (transfer minus own-fit) is paired by seed. The relevant noise is therefore the seed SD
of the gap, not the SD of either score on its own. The table shows that SD for one skip and for the mean gap over all of a
panel's single-block skips. AUROC, in pp.

In [8]:
import re

s5 = pd.concat([pd.read_csv(p) for p in sorted((ROOT / "experiments" / "4_4_without_target_data" / "results").glob("*.csv"))],
               ignore_index=True)
s5["H_s"] = s5["approx_layer"].map(lambda a: (lambda s: int(s[0][0]) if len(s) == 1 and int(s[0][1]) - int(s[0][0]) == 1 else None)(
    re.findall(r"\((\d+),\s*(\d+)\)", str(a))))
s5 = s5.dropna(subset=["H_s"]).drop_duplicates(["model", "dataset", "fit_dataset", "H_s", "seed"], keep="last")
s5 = s5[s5["dataset"] != "cifar100"]  # CIFAR-100 is only ever a donor
PARTNER = {"chestxray14_multilabel": "chestxray_pneumonia", "chestxray_pneumonia": "chestxray14_multilabel",
           "patchcamelyon": "tcga_ut", "tcga_ut": "patchcamelyon"}

rows = []
for head, col in [("retrained", "auroc"), ("frozen", "frozen_head_auroc")]:
    for (model, dataset), g in s5.groupby(["model", "dataset"]):
        own = g[g["fit_dataset"] == dataset].set_index(["seed", "H_s"])[col]
        for donor, label in [(PARTNER[dataset], "in-domain"), ("cifar100", "CIFAR-100")]:
            gap = (g[g["fit_dataset"] == donor].set_index(["seed", "H_s"])[col] - own).dropna() * 100
            if gap.empty:
                continue
            rows.append({"head": head, "donor": label, "model": model.split("/")[1], "dataset": dataset,
                         "mean gap": gap.groupby("H_s").mean().mean(),
                         "SD of mean gap": gap.groupby("seed").mean().std(),
                         "SD of one skip's gap": gap.groupby("H_s").std().median(),
                         "own-fit SD of one skip": own.groupby("H_s").std().median() * 100})
s5_noise = pd.DataFrame(rows).set_index(["head", "donor", "model", "dataset"]).sort_index()
display(s5_noise.round(2))
s5_noise.groupby(["head", "donor"]).agg({"mean gap": "mean", "SD of mean gap": "median",
                                         "SD of one skip's gap": "median", "own-fit SD of one skip": "median"}).round(2)

mean gap  \
head      donor     model                 dataset                            
frozen    CIFAR-100 dinov2-base           chestxray14_multilabel     -8.51   
                                          chestxray_pneumonia        -8.45   
                    phikon-v2             patchcamelyon              -0.30   
                                          tcga_ut                    -0.96   
                    rad-dino              chestxray14_multilabel     -0.37   
                                          chestxray_pneumonia        -0.15   
                    rad-dino-maira-2      chestxray14_multilabel     -0.83   
                                          chestxray_pneumonia         0.01   
                    vit-large-patch16-224 patchcamelyon              -2.84   
                                          tcga_ut                    -4.42   
          in-domain dinov2-base           chestxray14_multilabel     -0.79   
                                          chestxray_pneumonia        -0.47   
                    phikon-v2             patchcamelyon              -0.35   
                                          tcga_ut                    -0.00   
                    rad-dino              chestxray14_multilabel      0.00   
                                          chestxray_pneumonia        -0.04   
                    rad-dino-maira-2      chestxray14_multilabel      0.03   
                                          chestxray_pneumonia         0.29   
                    vit-large-patch16-224 patchcamelyon              -0.72   
                                          tcga_ut                    -0.47   
retrained CIFAR-100 dinov2-base           chestxray14_multilabel     -1.59   
                                          chestxray_pneumonia        -0.86   
                    phikon-v2             patchcamelyon              -0.01   
                                          tcga_ut                    -0.21   
                    rad-dino              chestxray14_multilabel      0.18   
                                          chestxray_pneumonia        -0.05   
                    rad-dino-maira-2      chestxray14_multilabel     -0.03   
                                          chestxray_pneumonia        -0.09   
                    vit-large-patch16-224 patchcamelyon              -0.34   
                                          tcga_ut                    -0.84   
          in-domain dinov2-base           chestxray14_multilabel     -0.17   
                                          chestxray_pneumonia        -0.13   
                    phikon-v2             patchcamelyon              -0.02   
                                          tcga_ut                     0.00   
                    rad-dino              chestxray14_multilabel      0.08   
                                          chestxray_pneumonia        -0.04   
                    rad-dino-maira-2      chestxray14_multilabel      0.07   
                                          chestxray_pneumonia         0.06   
                    vit-large-patch16-224 patchcamelyon              -0.18   
                                          tcga_ut                    -0.06   

                                                                  SD of mean gap  \
head      donor     model                 dataset                                  
frozen    CIFAR-100 dinov2-base           chestxray14_multilabel            0.19   
                                          chestxray_pneumonia               0.57   
                    phikon-v2             patchcamelyon                     0.09   
                                          tcga_ut                           0.02   
                    rad-dino              chestxray14_multilabel            0.02   
                                          chestxray_pneumonia               0.01   
                    rad-dino-maira-2      chestxray14_multilabel            0.17   
                                 

mean gap  SD of mean gap  SD of one skip's gap  \
head      donor                                                       
frozen    CIFAR-100     -2.68            0.18                  0.22   
          in-domain     -0.25            0.10                  0.09   
retrained CIFAR-100     -0.38            0.05                  0.17   
          in-domain     -0.04            0.04                  0.07   

                     own-fit SD of one skip  
head      donor                              
frozen    CIFAR-100                     0.2  
          in-domain                     0.2  
retrained CIFAR-100                     0.2  
          in-domain                     0.2

## Per-combination values

In [9]:
cols = [f"{m}_{k}" for m in METRICS for k in ("rho", "rholen", "dmg")]
display(nat[cols].round(2))
med[cols].round(2)

,angular_dist_rho,angular_dist_rholen,angular_dist_dmg,local_cka_rho,local_cka_rholen,local_cka_dmg,cka_embed_rho,cka_embed_rholen,cka_embed_dmg,cosine_sim_rho,...,cosine_sim_dmg,toast_linear_error_alltok_rho,toast_linear_error_alltok_rholen,toast_linear_error_alltok_dmg,toast_linear_error_alltok_heldout_rho,toast_linear_error_alltok_heldout_rholen,toast_linear_error_alltok_heldout_dmg,mse_rho,mse_rholen,mse_dmg
deitsmall_cifar100,0.96,0.93,0.17,0.81,0.71,0.53,0.91,0.91,0.17,0.83,...,0.68,0.92,0.89,-0.12,0.91,0.88,-0.12,0.85,0.68,0.68
deitsmall_imagenet1k,0.84,0.22,2.67,0.67,0.01,2.64,0.92,0.22,3.24,0.97,...,1.83,0.78,0.24,3.42,0.79,0.25,3.42,0.96,0.57,1.83
vitsmall_cifar100,0.96,0.58,3.12,0.42,-0.30,6.42,0.96,0.63,3.34,0.94,...,3.34,0.78,0.25,4.89,0.77,0.28,4.89,0.94,0.64,3.34
vitsmall_imagenet1k,0.96,0.91,3.26,0.13,-0.70,12.38,0.98,0.96,2.94,0.98,...,2.94,0.71,0.09,8.79,0.72,0.14,8.79,0.98,0.92,2.94
dinov2base_cifar100,0.86,0.14,1.90,0.89,0.44,2.83,0.89,0.82,1.35,0.90,...,1.35,0.70,0.16,4.80,0.69,0.16,4.80,0.90,0.86,1.35
dinov2base_imagenet1k,0.84,0.24,1.88,0.83,0.51,6.85,0.93,0.87,1.46,0.95,...,1.46,0.66,0.13,13.76,0.63,0.08,13.76,0.95,0.90,1.46
vitlarge_cifar100,0.68,0.55,0.19,0.75,0.58,0.03,0.76,0.41,0.17,0.75,...,0.17,0.78,0.63,0.12,0.78,0.64,0.12,0.75,0.39,0.17
vitlarge_imagenet1k,0.68,0.42,0.77,0.75,0.57,0.76,0.93,0.83,0.61,0.94,...,0.49,0.69,0.37,0.93,0.70,0.37,0.93,0.94,0.85,0.49


,angular_dist_rho,angular_dist_rholen,angular_dist_dmg,local_cka_rho,local_cka_rholen,local_cka_dmg,cka_embed_rho,cka_embed_rholen,cka_embed_dmg,cosine_sim_rho,...,cosine_sim_dmg,toast_linear_error_alltok_rho,toast_linear_error_alltok_rholen,toast_linear_error_alltok_dmg,toast_linear_error_alltok_heldout_rho,toast_linear_error_alltok_heldout_rholen,toast_linear_error_alltok_heldout_dmg,mse_rho,mse_rholen,mse_dmg
dinov2base_chestxray14multilabel,0.65,0.22,-0.38,0.75,0.50,-0.30,0.75,0.56,-0.22,0.77,...,-0.46,0.31,-0.11,0.06,0.31,-0.11,0.06,0.77,0.58,-0.46
dinov2base_chestxraypneumonia,0.65,-0.00,0.19,0.76,0.41,0.04,0.76,0.40,0.58,0.78,...,0.44,0.45,0.14,0.75,0.44,0.13,0.75,0.78,0.52,0.44
raddino_chestxray14multilabel,0.34,0.89,-0.33,0.25,0.75,-0.32,-0.53,-0.87,0.04,-0.56,...,0.04,0.32,0.90,-0.54,0.32,0.90,-0.54,-0.56,-0.84,0.04
raddino_chestxraypneumonia,0.46,0.22,-0.03,0.49,0.07,-0.03,0.49,0.07,-0.03,0.48,...,-0.03,0.43,0.13,-0.01,0.44,0.12,-0.01,0.50,-0.15,-0.03
raddinomaira2_chestxray14multilabel,0.49,0.59,-0.23,0.24,0.03,0.00,-0.14,-0.64,0.05,-0.16,...,0.05,0.53,0.72,-0.32,0.53,0.72,-0.32,-0.16,-0.64,0.05
raddinomaira2_chestxraypneumonia,0.60,0.73,0.04,0.30,-0.16,0.14,0.28,-0.53,0.18,0.21,...,0.18,0.57,0.55,0.07,0.57,0.55,0.07,0.21,-0.63,0.18
vitlarge_patchcamelyon,0.34,0.44,-0.11,0.57,0.64,-0.06,0.47,0.54,-0.06,0.46,...,-0.06,0.48,0.48,-0.06,0.48,0.48,-0.06,0.46,0.51,-0.06
vitlarge_tcgaut,0.09,0.28,-0.05,0.37,0.47,-0.09,0.17,0.36,-0.05,0.15,...,-0.05,0.28,0.31,-0.05,0.28,0.32,-0.05,0.15,0.34,-0.05
phikonv2_patchcamelyon,0.23,0.02,-0.14,0.26,0.07,-0.08,0.24,0.04,-0.08,0.20,...,-0.08,0.38,0.34,-0.18,0.38,0.33,-0.18,0.19,0.01,-0.08
phikonv2_tcgaut,0.73,0.17,0.02,0.76,0.35,0.01,0.87,0.69,0.00,0.88,...,0.00,0.75,0.31,0.02,0.76,0.34,0.02,0.88,0.75,0.00
